# 04 – Logistic Regression

Logistic Regression is the go-to starting point for **binary and multi-class classification**.

Topics covered:
1. Binary classification (Breast Cancer dataset)
2. Multi-class classification (Iris dataset)
3. Evaluation: confusion matrix, precision, recall, F1, ROC-AUC

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)

%matplotlib inline
sns.set_theme(style='whitegrid')
np.random.seed(42)

## 1. Binary Classification – Breast Cancer

In [ ]:
bc = load_breast_cancer()
X, y = bc.data, bc.target
print('Classes:', bc.target_names)
print('Shape:', X.shape)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc  = scaler.transform(X_te)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_tr_sc, y_tr)

y_pred = clf.predict(X_te_sc)
print(classification_report(y_te, y_pred, target_names=bc.target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=bc.target_names)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC Curve
y_prob = clf.predict_proba(X_te_sc)[:, 1]
fpr, tpr, _ = roc_curve(y_te, y_prob)
auc = roc_auc_score(y_te, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}', color='steelblue')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

## 2. Multi-class Classification – Iris

In [ ]:
iris = load_iris()
X_i, y_i = iris.data, iris.target

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_i, y_i, test_size=0.2, random_state=42, stratify=y_i)

sc_i = StandardScaler()
X_tr_i_sc = sc_i.fit_transform(X_tr_i)
X_te_i_sc  = sc_i.transform(X_te_i)

clf_mc = LogisticRegression(multi_class='auto', max_iter=1000, random_state=42)
clf_mc.fit(X_tr_i_sc, y_tr_i)

y_pred_i = clf_mc.predict(X_te_i_sc)
print(classification_report(y_te_i, y_pred_i, target_names=iris.target_names))

In [ ]:
# Confusion matrix for Iris
cm_i = confusion_matrix(y_te_i, y_pred_i)
disp_i = ConfusionMatrixDisplay(cm_i, display_labels=iris.target_names)
disp_i.plot(cmap='Greens')
plt.title('Iris – Confusion Matrix')
plt.show()

## 3. Decision Boundary Visualisation (2 features)

We use only the first two Iris features for plotting.

In [ ]:
from matplotlib.colors import ListedColormap

X2 = iris.data[:, :2]   # sepal length & sepal width
y2 = iris.target

clf2 = LogisticRegression(max_iter=1000)
clf2.fit(X2, y2)

x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = clf2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

cmap_bg = ListedColormap(['#ffaaaa', '#aaffaa', '#aaaaff'])
cmap_pt = ListedColormap(['red', 'green', 'blue'])

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=cmap_bg, alpha=0.6)
scatter = plt.scatter(X2[:, 0], X2[:, 1], c=y2, cmap=cmap_pt, edgecolors='k', s=40)
plt.xlabel('Sepal Length'); plt.ylabel('Sepal Width')
plt.title('Logistic Regression – Decision Boundaries')
plt.legend(handles=scatter.legend_elements()[0], labels=iris.target_names)
plt.show()

## 4. Key Takeaways

| Concept | Notes |
|---------|-------|
| Sigmoid / Softmax | Squashes output to probability |
| Precision | TP / (TP + FP) – how trustworthy positives are |
| Recall | TP / (TP + FN) – how many positives are found |
| F1-score | Harmonic mean of precision & recall |
| ROC-AUC | 1 = perfect; 0.5 = random |

**Next:** `02_Intermediate/01_Decision_Trees_Random_Forests.ipynb`